# Coronary CAC and obstructive-lesion simulation — overview

NHANES does not measure coronary artery calcium (CAC, Agatston score) or
coronary angiographic lesions, so a simulation that needs these
attributes must impute them. This project gives a two-step recipe:

1. **Risk factors → CAC.** A two-part log-normal model: a logistic
   regression for `P(CAC > 0)` and a normal regression for
   `log(CAC + 1) | CAC > 0`. Coefficients are anchored to MESA
   reference distributions (McClelland 2006, Circulation) and the MESA
   10-year CHD risk model (McClelland 2015, JACC).
2. **CAC → obstructive lesion.** A logistic on `log(CAC + 1)`,
   calibrated to CONFIRM-registry / Budoff JACC 2007 prevalence
   anchors of >=50% angiographic stenosis by CAC category.

Both steps return *distributions* the simulation can sample from rather
than point predictions, so simulants inherit realistic between-person
variability.

## Why two steps and not a direct lesion model

Angiographic stenosis is almost never measured in unselected
populations — patients have to be sick enough to get referred to a cath
lab. Any direct "risk factors → stenosis" model trained on cath-lab
data carries severe selection bias. CAC, in contrast, has been
measured in unselected cohorts (MESA, CARDIA, Heinz Nixdorf) and has a
well-characterized relationship with both risk factors (upstream) and
obstructive CAD (downstream). Going through CAC keeps both edges
well-supported by data.

## Headline results

Sampling 50,000 60-year-old non-Hispanic white men with average risk
factors using the default parameters in `cac_model.py`:

| Quantile         | Sampled | MESA reference (Circulation 2006) |
| :---             |    ---: |                              ---: |
| P(CAC > 0)       |    0.81 |                            ~ 0.75 |
| Median CAC       |    ~65  |                             ~ 30  |
| 75th pct CAC     |    ~232 |                             ~ 180 |
| 90th pct CAC     |    ~640 |                             ~ 620 |

The model is within ~10% of MESA percentiles at older ages and slightly
overpredicts at younger ages; tune `CACModelParams.beta_intercept` or
refit on individual MESA data to tighten. See `01_cac_distribution.ipynb`
for the full validation grid.

## Files

- `cac_model.py` — two-part log-normal CAC model
- `cad_lesion_model.py` — CAC → obstructive-lesion logistic
- `01_cac_distribution.ipynb` — calibrate, validate, and visualize
  the CAC distribution against MESA reference tables
- `02_cac_to_lesion.ipynb` — apply the two-step pipeline to a
  synthetic NHANES-style cohort and show the resulting lesion
  prevalence by age and sex

## How to plug into a simulation

```python
from cac_model import sample_cac
from cad_lesion_model import sample_obstructive_lesion

# population_df has columns: age, male, race, smoke, dm,
# sbp, bmi, tc, hdl, bp_med, lipid_med
cac = sample_cac(population_df, rng=rng)
lesion = sample_obstructive_lesion(cac, rng=rng)
```

Both samplers accept an `rng` for reproducibility and a `params`
object so you can swap in your own coefficients. If you have access
to individual MESA data, refit `CACModelParams` directly.

## Limitations

- Coefficients are taken from published literature, not refit on
  individual data. Treat the absolute numbers as illustrative.
- The CAC → lesion mapping is fit to a *referred* population
  (CONFIRM). In a general-population cohort, scale `offset` down
  by ~0.7-1.0 logits to match expected prevalence.
- CAC is age-cumulative; for a longitudinal simulation, you'll want
  a *progression* model on top of this *prevalence* model.
